# V12 Self-Play (Colab L4/T4)

**Goal:** generate ~400 V12 self-play games to anchor the steady-state distribution for the V12 policy distillation corpus. The bulk of V12's training signal comes from crisis mining (run separately on M5); this notebook contributes the steady-state coverage.

Player: **pillar2y2_epoch_40 + value_head_v11 + q_weight=2.0**, the strongest known player (Phase 3R: mean 13,476 / median 16,427 / 69% ≥15K on M5 pilot).

Settings (mirror M5 pilot exactly):
- 400 sims, q_weight=2.0, batch_size=8 (HISTORY lesson 115)
- max-turns 8000 (cap-hits dominate but visit distributions still inform)
- temperature-moves 15 (standard exploration)
- Dirichlet α=0.3, weight=0.25 (defaults)

**Upload to `MyDrive/alphatrain/` before running:**
1. `colorlines_selfplay_train.tar.gz` (built by `scripts/build_selfplay_tarball.sh`)
2. `pillar2y2_epoch_40.pt` (~143 MB)
3. `value_head_v11.pt` (~38 KB)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
DRIVE = '/content/drive/MyDrive/alphatrain'

!cp {DRIVE}/colorlines_selfplay_train.tar.gz /content/
!cd /content && tar xzf colorlines_selfplay_train.tar.gz

os.makedirs('/content/alphatrain/data', exist_ok=True)
!cp {DRIVE}/pillar2y2_epoch_40.pt /content/alphatrain/data/pillar2y2_epoch_40.pt
!cp {DRIVE}/value_head_v11.pt /content/alphatrain/data/value_head_v11.pt
print(f'Model: {os.path.getsize("/content/alphatrain/data/pillar2y2_epoch_40.pt")/1e6:.0f} MB')
print(f'Value head: {os.path.getsize("/content/alphatrain/data/value_head_v11.pt")/1e3:.0f} KB')

!pip install -q numpy numba scipy

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    g = torch.cuda.get_device_properties(0)
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {g.total_memory / 1e9:.1f} GB')

In [ ]:
# === V12 Self-Play (anchor the steady-state distribution) ===
# Distribute by editing SEED_START / SEED_END if running multiple instances.
# Local M5 pilot used 1200000-1200099. Use disjoint range here.
SEED_START = 1200100
SEED_END = 1200500   # 400 games
# ============================================================

SIMS = 400               # matches the M5 pilot, target-range player
Q_WEIGHT = 2.0           # calibrated for value_head_v11 (HISTORY 133)
WORKERS = 24             # Colab CPU
BATCH_SIZE = 8           # HISTORY lesson 115 — bs=8 is essential
MAX_TURNS = 8000         # cap-hits cluster at ~16,400-16,700 (~2.05/turn)
TEMP_MOVES = 15          # standard exploration window
SAVE_DIR = f'{DRIVE}/selfplay_v12'

os.makedirs(SAVE_DIR, exist_ok=True)
print(f'Seeds: {SEED_START}-{SEED_END-1} ({SEED_END-SEED_START} games)')
print(f'Sims: {SIMS}, q_weight: {Q_WEIGHT}, Cap: {MAX_TURNS} turns')
print(f'Workers: {WORKERS}, bs: {BATCH_SIZE}')
print(f'Save: {SAVE_DIR}')

!python -m alphatrain.scripts.selfplay \
    --model /content/alphatrain/data/pillar2y2_epoch_40.pt \
    --value-head-path /content/alphatrain/data/value_head_v11.pt \
    --q-weight {Q_WEIGHT} \
    --seed-start {SEED_START} --seed-end {SEED_END} \
    --sims {SIMS} --batch-size {BATCH_SIZE} \
    --save-dir {SAVE_DIR} \
    --workers {WORKERS} --device cuda \
    --temperature-moves {TEMP_MOVES} \
    --max-turns {MAX_TURNS} \
    --compile

## Notes

- **Resume support:** if Colab disconnects, just re-run the previous cell. It skips seeds whose game files already exist in `SAVE_DIR`.
- **Wall time on L4:** ~30-60s per game in 24 workers parallel → ~6-10 hours for 400 games. T4 may be 1.5-2× slower.
- **Watch:** the GPU server prints `[GPU] X evals, Y fwd (avg bs=Z, ZZZZ evals/s)` every 10K forwards. avg bs should be ≥50 and evals/s in the thousands; if either drops, MPS/CUDA is bottlenecked or workers are stalled.
- **Output:** each game saved as `game_seed{N}_score{S}.json` in the save dir. The build script (run on M5) merges these into the V12 training tensor.